<a href="https://colab.research.google.com/github/demichie/Principles-of-Numerical-Modelling-in-Geosciences/blob/main/Chapter6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6: Numerical Solution of the 1D Heat (Diffusion) Equation

This notebook contains the Python code examples for Chapter 6 of "Principles of Numerical Modelling in Geosciences".

In the previous chapter, we established the physical and mathematical foundations for the Heat (or Diffusion) Equation. For a one-dimensional domain, assuming constant thermal diffusivity $\kappa$ and no internal heat sources, this equation is:
$$ \frac{\partial T}{\partial t} = \kappa \frac{\partial^2 T}{\partial x^2} $$
This chapter is dedicated to developing and implementing a numerical method to solve this fundamental parabolic PDE. The process involves discretizing both time and space and, crucially, specifying **boundary conditions** to obtain a unique, physically relevant solution.

## 6.2 Numerical Discretization of the 1D Heat Equation

To solve the 1D Heat Equation numerically, we discretize both the spatial and time domains and approximate the partial derivatives using finite differences.

### 6.2.3 The Forward-Time Central-Space (FTCS) Scheme: Formulation

We will use a **Forward Difference in Time** for the time derivative and a **Central Difference in Space** for the second spatial derivative. Combining these approximations leads to the **Forward-Time Central-Space (FTCS)** scheme.

The update formula for the temperature $T_i^{n+1}$ at node $i$ and time step $n+1$ is:
$$ T_i^{n+1} = T_i^n + \alpha \left( T_{i+1}^n - 2T_i^n + T_{i-1}^n \right) $$
where $\alpha$ is a dimensionless parameter defined as:
$$ \alpha = \kappa \frac{\Delta t}{(\Delta x)^2} $$
This equation is applied to all *interior* grid points. The temperatures at the boundary points are determined by the specific boundary conditions.

## 6.3 Python Implementation and Results for the 1D Heat Equation (FTCS)

We will now implement the FTCS scheme in Python to simulate 1D heat diffusion. This will allow us to observe its behaviour under different conditions, particularly concerning stability and the application of boundary conditions.

### 6.3.1 Scenario 1: Dirichlet Boundary Conditions

**Problem Setup:**
*   **Domain:** A 1D rod of length $L = 1.0$ m.
*   **Grid:** $N_x = 51$ points, so $\Delta x = 0.02$ m.
*   **Property:** Thermal diffusivity $\kappa = 1.0 \times 10^{-6} \text{ m}^2/\text{s}$.
*   **Initial Condition:** A "hot pulse" of $100^\circ$C between $x=0.4$ and $x=0.6$, with an ambient temperature of $20^\circ$C elsewhere.
*   **Boundary Conditions:** Fixed (Dirichlet) temperatures of $T=20^\circ$C at both ends.

**Python Implementation:**
The following script simulates this scenario. To avoid storing data at every time step, it uses the **modulo operator (%)** to save the temperature profile for plotting only at regular intervals. This useful operator will be discussed in more detail in the next chapter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
lengthDomain = 1.0  # m
numXPoints = 51     # Number of spatial grid points
dxStep = lengthDomain / (numXPoints - 1)
xGrid = np.linspace(0, lengthDomain, numXPoints)

kappaDiffusivity = 1.0e-6  # Thermal diffusivity (m^2/s)

# Time parameters and Stability
# alpha = kappa * dt / dx^2. For stability, alpha <= 0.5
alphaTarget = 0.45 # Choose a value for alpha (e.g., 0.45 for stable, 0.55 for unstable)
# alphaTarget = 0.55 # Uncomment to test instability

dtStep = alphaTarget * dxStep**2 / kappaDiffusivity # Calculate dt based on target alpha
tFinal = 100000.0  # s (e.g., ~27 hours)
numTSteps = int(tFinal / dtStep)

print(f"Domain Length (L): {lengthDomain} m")
print(f"Number of X Points (numXPoints): {numXPoints}")
print(f"Spatial Step (dxStep): {dxStep:.4f} m")
print(f"Thermal Diffusivity (kappaDiffusivity): {kappaDiffusivity:.1e} m^2/s")
print(f"Target Alpha (alphaTarget): {alphaTarget:.2f}")
print(f"Time Step (dtStep): {dtStep:.2f} s")
print(f"Final Time (tFinal): {tFinal:.1f} s")
print(f"Number of Time Steps (numTSteps): {numTSteps}")
print(f"Calculated alpha = {kappaDiffusivity * dtStep / dxStep**2:.3f}") # Verify actual alpha

# --- Initial Condition ---
temperatureCurrent = np.ones(numXPoints) * 20.0  # Ambient temperature
pulseStartIndex = int(0.4 * lengthDomain / dxStep)
pulseEndIndex = int(0.6 * lengthDomain / dxStep)
temperatureCurrent[pulseStartIndex : pulseEndIndex + 1] = 100.0

# Store temperature profiles for plotting
plotIntervalRatio = 0.1 # Plot approx 10 intermediate profiles
plotIntervalSteps = max(1, int(numTSteps * plotIntervalRatio))
timePointsToPlot = [0.0]
tempProfilesToPlot = [temperatureCurrent.copy()]

# --- Time-stepping loop (FTCS) ---
temperatureOld = temperatureCurrent.copy() # For T^n values

for n in range(1, numTSteps + 1):
    # Update interior points using values from T_old (time level n)
    for i in range(1, numXPoints - 1):
        temperatureCurrent[i] = temperatureOld[i] + \
            alphaTarget * (temperatureOld[i+1] - 2*temperatureOld[i] + temperatureOld[i-1])

    # Apply Dirichlet Boundary Conditions (temperature remains T^n+1 after update)
    temperatureCurrent[0] = 20.0   # Fixed temperature at x=0
    temperatureCurrent[numXPoints-1] = 20.0 # Fixed temperature at x=L

    # Update temperatureOld for the next iteration
    temperatureOld = temperatureCurrent.copy()

    # Store profile for plotting at specified intervals
    if n % plotIntervalSteps == 0 or n == numTSteps:
        timePointsToPlot.append(n * dtStep)
        tempProfilesToPlot.append(temperatureCurrent.copy())

# --- Plotting Results ---
plt.figure(figsize=(10, 7))
for iPlot in range(len(tempProfilesToPlot)):
    plt.plot(xGrid, tempProfilesToPlot[iPlot],
             label=f't = {timePointsToPlot[iPlot]:.0f} s')

plt.xlabel("Position x [m]")
plt.ylabel("Temperature [deg C]")
plt.title(f"1D Heat Diffusion (FTCS, Dirichlet, $\kappa$={kappaDiffusivity:.1e}, $\\alpha$={alphaTarget:.2f})")
plt.legend(loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.grid(True)
plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout for legend outside
plt.show()

### 6.3.2 Stability Analysis of the FTCS Scheme

The FTCS scheme is **conditionally stable**. For a stable solution, the parameter $\alpha$ must satisfy:
$$ \alpha = \kappa \frac{\Delta t}{(\Delta x)^2} \leq \frac{1}{2} $$
Violating this condition (e.g., by choosing `alphaTarget = 0.55` in the code above) leads to unphysical, growing oscillations, as the solution becomes unstable.

### 6.3.3 Scenario 2: Implementing a Neumann Boundary Condition

We now adapt the simulation to handle a zero-flux (insulated) **Neumann boundary condition** at $x=0$. The initial hot pulse is placed adjacent to this boundary to observe its effect.

The update formula for the insulated node $i=0$ is derived using a "ghost point" and becomes:
$$ T_0^{n+1} = T_0^n + 2\alpha (T_1^n - T_0^n) $$
The following code snippet shows the key modifications inside the time-stepping loop.

In [ ]:
# This cell shows the core logic for the Neumann BC case.
# A full runnable script is provided in the exercises section.
import numpy as np
import matplotlib.pyplot as plt

# Assume parameters are defined as in the Dirichlet case (alphaTarget=0.45)
lengthDomain = 1.0
numXPoints = 51
dxStep = lengthDomain / (numXPoints - 1)
xGrid = np.linspace(0, lengthDomain, numXPoints)
kappaDiffusivity = 1.0e-6
alphaTarget = 0.45
dtStep = alphaTarget * dxStep**2 / kappaDiffusivity
tFinal = 100000.0
numTSteps = int(tFinal / dtStep)

# --- New Initial Condition for Neumann case ---
temperatureCurrent = np.ones(numXPoints) * 20.0
pulseEndIndex = int(0.2 * lengthDomain / dxStep) # Pulse near x=0
temperatureCurrent[0 : pulseEndIndex + 1] = 100.0

tempProfilesToPlot_N = [temperatureCurrent.copy()]
timePointsToPlot_N = [0.0]
plotIntervalSteps = max(1, int(numTSteps * 0.1))

# --- Time-stepping loop with Neumann BC ---
temperatureOld = temperatureCurrent.copy()

for n in range(1, numTSteps + 1):
    # Update interior points (i = 1 to Nx-2)
    for i in range(1, numXPoints - 1):
        temperatureCurrent[i] = temperatureOld[i] + \
            alphaTarget * (temperatureOld[i+1] - 2*temperatureOld[i] + temperatureOld[i-1])

    # --- Apply Boundary Conditions ---
    # Apply Neumann BC at x=0 (node i=0)
    temperatureCurrent[0] = temperatureOld[0] + \
                         2 * alphaTarget * (temperatureOld[1] - temperatureOld[0])

    # Apply Dirichlet BC at x=L (node i=Nx-1)
    temperatureCurrent[numXPoints-1] = 20.0

    temperatureOld = temperatureCurrent.copy()

    # Store for plotting
    if n % plotIntervalSteps == 0 or n == numTSteps:
        timePointsToPlot_N.append(n * dtStep)
        tempProfilesToPlot_N.append(temperatureCurrent.copy())

# --- Plotting Results ---
plt.figure(figsize=(10, 7))
for iPlot in range(len(tempProfilesToPlot_N)):
    plt.plot(xGrid, tempProfilesToPlot_N[iPlot],
             label=f't = {timePointsToPlot_N[iPlot]:.0f} s')

plt.xlabel("Position x [m]")
plt.ylabel("Temperature [deg C]")
plt.title(f"1D Heat Diffusion (FTCS, Neumann at x=0, $\alpha$={alphaTarget:.2f})")
plt.legend(loc='best')
plt.grid(True)
plt.show()

## Chapter 6 Exercises

The following cells provide the solutions for the exercises in Chapter 6.

### E6.1: Exploring the Stability Parameter $\alpha$

In [ ]:
# E6.1: Exploring the Stability Parameter alpha
import numpy as np
import matplotlib.pyplot as plt

def run_heat_equation(alphaTarget):
    """A helper function to run the simulation for a given alpha."""
    lengthDomain = 1.0
    numXPoints = 51
    dxStep = lengthDomain / (numXPoints - 1)
    xGrid = np.linspace(0, lengthDomain, numXPoints)
    kappaDiffusivity = 1.0e-6
    tFinal = 100000.0

    dtStep = alphaTarget * dxStep**2 / kappaDiffusivity
    if dtStep == 0: # Avoid division by zero if alpha is zero
        numTSteps = 0
    else:
        numTSteps = int(tFinal / dtStep)

    print(f"\nRunning for alpha = {alphaTarget:.2f}, dt = {dtStep:.2f} s, num_steps = {numTSteps}")

    # Initial and Boundary Conditions
    temperature = np.ones(numXPoints) * 20.0
    pulseStartIndex = int(0.4 * lengthDomain / dxStep)
    pulseEndIndex = int(0.6 * lengthDomain / dxStep)
    temperature[pulseStartIndex : pulseEndIndex + 1] = 100.0

    initial_temp = temperature.copy()

    # Time-stepping loop
    temperatureOld = temperature.copy()
    for n in range(1, numTSteps + 1):
        for i in range(1, numXPoints - 1):
            temperature[i] = temperatureOld[i] + \
                alphaTarget * (temperatureOld[i+1] - 2*temperatureOld[i] + temperatureOld[i-1])
        temperature[0] = 20.0
        temperature[-1] = 20.0
        temperatureOld = temperature.copy()

    return xGrid, initial_temp, temperature

# Run simulations for different alpha values
alphas_to_test = [0.45, 0.50, 0.51, 0.60]
results = {}
for alpha in alphas_to_test:
    x, T_initial, T_final = run_heat_equation(alpha)
    results[alpha] = T_final

# Plotting the results
plt.figure(figsize=(10, 7))
plt.plot(x, T_initial, 'k:', label='Initial Condition')
colors = ['blue', 'green', 'orange', 'red']
for i, alpha in enumerate(alphas_to_test):
    plt.plot(x, results[alpha], color=colors[i], label=f'Final T ($\alpha$ = {alpha:.2f})')

plt.title("E6.1: Exploring the Stability Parameter $\alpha$")
plt.xlabel("Position x [m]")
plt.ylabel("Temperature [deg C]")
plt.legend()
plt.grid(True)
plt.show()

### E6.2: Impact of Thermal Diffusivity $\kappa$

In [ ]:
# E6.2: Impact of Thermal Diffusivity kappa
import numpy as np
import matplotlib.pyplot as plt

def run_heat_equation_kappa(kappa, alphaTarget=0.45):
    """Helper function to run simulation for a given kappa."""
    lengthDomain = 1.0
    numXPoints = 51
    dxStep = lengthDomain / (numXPoints - 1)
    xGrid = np.linspace(0, lengthDomain, numXPoints)
    tFinal = 100000.0

    dtStep = alphaTarget * dxStep**2 / kappa
    numTSteps = int(tFinal / dtStep)

    print(f"\nRunning for kappa = {kappa:.1e} m^2/s")
    print(f"   Time step dt = {dtStep:.2f} s")

    temperature = np.ones(numXPoints) * 20.0
    pulseStartIndex = int(0.4 * lengthDomain / dxStep)
    pulseEndIndex = int(0.6 * lengthDomain / dxStep)
    temperature[pulseStartIndex : pulseEndIndex + 1] = 100.0

    initial_temp = temperature.copy()

    temperatureOld = temperature.copy()
    for n in range(1, numTSteps + 1):
        for i in range(1, numXPoints - 1):
            temperature[i] = temperatureOld[i] + \
                alphaTarget * (temperatureOld[i+1] - 2*temperatureOld[i] + temperatureOld[i-1])
        temperature[0] = 20.0
        temperature[-1] = 20.0
        temperatureOld = temperature.copy()

    return xGrid, initial_temp, temperature

# Run for different kappa values
kappa_slow = 1.0e-7
kappa_fast = 5.0e-6

x_slow, T0_slow, Tf_slow = run_heat_equation_kappa(kappa_slow)
x_fast, T0_fast, Tf_fast = run_heat_equation_kappa(kappa_fast)

# Plotting
plt.figure(figsize=(10, 7))
plt.plot(x_slow, T0_slow, 'k:', label='Initial Condition')
plt.plot(x_slow, Tf_slow, 'r-', label=f'Final T ($\kappa$ = {kappa_slow:.1e}) - Slow Diffusion')
plt.plot(x_fast, Tf_fast, 'b-', label=f'Final T ($\kappa$ = {kappa_fast:.1e}) - Fast Diffusion')
plt.title("E6.2: Impact of Thermal Diffusivity $\kappa$")
plt.xlabel("Position x [m]")
plt.ylabel("Temperature [deg C]")
plt.legend()
plt.grid(True)
plt.show()

### E6.3: Implementing a Different Neumann Boundary Condition

In [ ]:
# E6.3: Constant Heat Flux Neumann BC
import numpy as np
import matplotlib.pyplot as plt

# Parameters
lengthDomain = 1.0
numXPoints = 51
dxStep = lengthDomain / (numXPoints - 1)
xGrid = np.linspace(0, lengthDomain, numXPoints)

kappa = 1.0e-6
kc = 2.0  # W / (m K)
q0 = 10.0 # W / m^2

alphaTarget = 0.45
dtStep = alphaTarget * dxStep**2 / kappa
tFinal = 200000.0
numTSteps = int(tFinal / dtStep)

# 1. Derive Update Formula
# dT/dx = -q0/kc
# (T1 - T-1)/(2*dx) = -q0/kc  => T-1 = T1 + 2*dx*q0/kc
# T0_n+1 = T0_n + alpha*(T1_n - 2*T0_n + T-1_n)
# T0_n+1 = T0_n + alpha*(T1_n - 2*T0_n + T1_n + 2*dx*q0/kc)
# T0_n+1 = T0_n + 2*alpha*(T1_n - T0_n) + 2*alpha*dx*q0/kc
# T0_n+1 = T0_n + 2*alpha*(T1_n - T0_n) + 2*(k*dt/dx^2)*dx*q0/kc
# T0_n+1 = T0_n + 2*alpha*(T1_n - T0_n) + (2*k*dt*q0)/(dx*kc)
neumann_flux_term = (2 * kappa * dtStep * q0) / (dxStep * kc)

# Initial Condition & Storage
temperature = np.ones(numXPoints) * 20.0
tempProfiles = [temperature.copy()]
timePoints = [0.0]
plotIntervalSteps = max(1, int(numTSteps * 0.1))

# 2. Implement and Simulate
temperatureOld = temperature.copy()
for n in range(1, numTSteps + 1):
    # Interior points
    for i in range(1, numXPoints - 1):
        temperature[i] = temperatureOld[i] + alphaTarget * (temperatureOld[i+1] - 2*temperatureOld[i] + temperatureOld[i-1])

    # Boundary Conditions
    # Neumann BC at x=0 (constant flux)
    temperature[0] = temperatureOld[0] + 2 * alphaTarget * (temperatureOld[1] - temperatureOld[0]) + neumann_flux_term
    # Dirichlet BC at x=L
    temperature[-1] = 20.0

    temperatureOld = temperature.copy()

    if n % plotIntervalSteps == 0 or n == numTSteps:
        timePoints.append(n * dtStep)
        tempProfiles.append(temperature.copy())

# 3. Plot and Discuss
plt.figure(figsize=(10, 7))
for i in range(len(tempProfiles)):
    plt.plot(xGrid, tempProfiles[i], label=f't = {timePoints[i]:.0f} s')

plt.title("E6.3: Constant Heat Flux at x=0")
plt.xlabel("Position x [m]")
plt.ylabel("Temperature [deg C]")
plt.legend()
plt.grid(True)
plt.show()

# The system reaches a steady state where the temperature profile is linear.
# At steady state, the conductive flux must be constant everywhere to balance the influx q0.
# Since q = -kc * dT/dx = q0, dT/dx must be a constant (-q0/kc), which implies a linear profile.

### E6.4: Grid Refinement Study (Qualitative)

In [ ]:
# E6.4: Grid Refinement Study
import numpy as np
import matplotlib.pyplot as plt

def run_heat_grid_refinement(numXPoints, alphaTarget=0.45):
    """Helper function for the grid refinement study."""
    lengthDomain = 1.0
    dxStep = lengthDomain / (numXPoints - 1)
    xGrid = np.linspace(0, lengthDomain, numXPoints)
    kappaDiffusivity = 1.0e-6
    tFinal = 100000.0

    dtStep = alphaTarget * dxStep**2 / kappaDiffusivity
    numTSteps = int(tFinal / dtStep)

    print(f"\nRunning for Nx = {numXPoints}")
    print(f"   dx = {dxStep:.4f} m, dt = {dtStep:.2f} s, num_steps = {numTSteps}")

    temperature = np.ones(numXPoints) * 20.0
    pulseStartIndex = int(0.4 * lengthDomain / dxStep)
    pulseEndIndex = int(0.6 * lengthDomain / dxStep)
    temperature[pulseStartIndex : pulseEndIndex + 1] = 100.0

    temperatureOld = temperature.copy()
    for n in range(1, numTSteps + 1):
        for i in range(1, numXPoints - 1):
            temperature[i] = temperatureOld[i] + \
                alphaTarget * (temperatureOld[i+1] - 2*temperatureOld[i] + temperatureOld[i-1])
        temperature[0] = 20.0
        temperature[-1] = 20.0
        temperatureOld = temperature.copy()

    return xGrid, temperature

# Run for different grid resolutions
nx_vals = [21, 51, 101]
results = {}
for nx in nx_vals:
    x, T_final = run_heat_grid_refinement(nx)
    results[nx] = (x, T_final)

# Plotting
plt.figure(figsize=(10, 7))
for nx in nx_vals:
    x, T_final = results[nx]
    plt.plot(x, T_final, 'o-', markersize=4, label=f'Nx = {nx}')

plt.title("E6.4: Grid Refinement Study")
plt.xlabel("Position x [m]")
plt.ylabel("Final Temperature (t=100,000s) [deg C]")
plt.legend()
plt.grid(True)
plt.show()